## Notebook Workflow Structure

This notebook systematically tests Epic clinical notes appointments extraction and retrieval functionality:

---
**Cells/Workflow Order:**

| Cell # | Purpose | Expected Results |
|--------|---------|------------------|
| 1-2 | Setup (imports, random seed) | Environment configured |
| 3 | Cleanup previous outputs | No stale data remains |
| 4 | Start ES container + credentials | Docker running, credentials file generated |
| 5-7 | Populate dummy patient data + ingest Epic clinical notes appointments | Documents in Elasticsearch |
| 8 | Index refresh verification | All indices have documents |
| 9-10 | Initialize database and logger | SQLite DB created |
| 11-12 | Create pat2vec config with epic_clinical_notes_appointments mode | Config with correct options |
| 13-14 | Run pat2vec pipeline | Pipeline processes patients successfully |
| 15-16 | Extract all features from database | Features DataFrame populated |
| 17 | Table error detection (NEW) | Verify no table errors from pat_maker output |
| 18-19 | Merge builder functionality (ENHANCED) | Merge function with full validation |
| 20 | Final verification | All assertions pass, TEST SUCCESSFUL |

---
**Test Failure Conditions:**
- Any cell raises unhandled exception
- Elasticsearch container fails to start
- No patient IDs generated after population
- Empty DataFrame from feature extraction
- Table errors in pat_maker output (e.g., "no such table", "OperationalError")

In [ ]:
import os
import shutil
import sys

from pat2vec.util.post_processing_build_methods import build_merged_epr_mct_annot_df

import pandas as pd

In [ ]:
current_dir = os.getcwd()
grandparent_dir = os.path.dirname(os.path.dirname(current_dir))

sys.path.insert(0, os.path.join(grandparent_dir, "pat2vec"))
sys.path.append(grandparent_dir)
pat2vec_dir = os.path.abspath(os.path.join(grandparent_dir, "pat2vec"))
sys.path.insert(0, pat2vec_dir)

print(f"Pat2vec path: {pat2vec_dir}")

In [ ]:
for dir_to_remove in ["epic_clinical_notes_appointments_test_project"]:
    try:
        shutil.rmtree(dir_to_remove, ignore_errors=True)
    except Exception as e:
        raise RuntimeError(
            f"Failed to clean up '{dir_to_remove}' directory: {e}. "
            "Critical error - cannot start with stale data.",
        ) from e

print("Previous outputs cleaned.")

In [ ]:
from pat2vec.util.docker_elastic import ElasticContainer

es_container = ElasticContainer()
es_container.stop()

print("Starting Elasticsearch container (this may take a few seconds)...")
if not es_container.start():
    raise RuntimeError(
        "Failed to start Elasticsearch container. Check if Docker is running.",
    )

host, username, password = es_container.get_credentials()

creds_filename = "test_elastic_credentials.py"
creds_content = f"""
username = "{username}"
password = "{password}"
api_key = None
hosts = ["{host}"]
"""

with open(creds_filename, "w") as f:
    f.write(creds_content)

print(f"Created '{creds_filename}' pointing to test cluster at {host}")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

schema_path = os.path.abspath("test_files/elastic_schemas.json")
config_populate = config_class(
    proj_name="epic_clinical_notes_appointments_test_project",
    credentials_path=creds_filename,
    test_schema_path=schema_path,
    testing=True,
    testing_elastic=True,
    global_start_year=2020,
    global_start_month=1,
    global_start_day=1,
    global_end_year=2023,
    global_end_month=12,
    global_end_day=31,
)

In [ ]:
from pat2vec.util.get_dummy_data_cohort_searcher import populate_elastic_with_dummy_data

print("Populating test Elasticsearch cluster with dummy data...")
patient_ids = populate_elastic_with_dummy_data(config_populate, n_patients=5)

print()
print("Population complete.")
print(f"Generated {len(patient_ids)} dummy patients.")
print(f"Patient IDs: {patient_ids}")

In [ ]:
from pat2vec.pat2vec_search.cogstack_search_methods import initialize_cogstack_client

cs = initialize_cogstack_client(config_populate)

indices = [
    "epr_documents",
    "basic_observations",
    "observations",
    "order",
    "pims_apps",
]
print("Refreshing indices...")
cs.elastic.indices.refresh(index=indices, ignore_unavailable=True)
print("Indices refreshed.")

In [ ]:
from pat2vec.util.elasticsearch_methods import ingest_data_to_elasticsearch
from pat2vec.util.get_dummy_data_cohort_searcher import (
    generate_epic_clinical_notes_appointments_data,
)

epic_app_dfs = []
for pid in patient_ids:
    df = generate_epic_clinical_notes_appointments_data(
        num_rows=3,
        entered_list=[pid],
        global_start_year=int(config_populate.global_start_year),
        global_start_month=int(config_populate.global_start_month),
        global_end_year=int(config_populate.global_end_year),
        global_end_month=int(config_populate.global_end_month),
    )
    epic_app_dfs.append(df)

df_epic_app = (
    pd.concat(epic_app_dfs, ignore_index=True)
    if len(epic_app_dfs) > 1
    else epic_app_dfs[0]
)
df_epic_app = df_epic_app.where(pd.notnull(df_epic_app), None)

ingest_data_to_elasticsearch(
    df_epic_app,
    "epic_clinical_notes_appointments",
    es_client=cs.elastic,
)
cs.elastic.indices.refresh(index="epic_clinical_notes_appointments")

print(
    f"Ingested {len(df_epic_app)} Epic clinical notes appointments records for {len(patient_ids)} patients",
)

In [ ]:
PROJ_NAME = "epic_clinical_notes_appointments_test_project"
DB_FILENAME = "temp_epic_clinical_notes_appointments_db.sqlite"
DB_PATH = os.path.join(PROJ_NAME, "outputs", DB_FILENAME)

os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
except Exception as e:
    raise RuntimeError(
        f"Failed to remove old database file '{DB_PATH}': {e}. "
        "Critical error - cannot start with stale data.",
    ) from e

db_connection_string = f"sqlite:///{DB_PATH}"
print(f"Database connection string set to: {db_connection_string}")

In [ ]:
from pat2vec.util.logger_setup import setup_logger

logger = setup_logger()
print("Logger initialized.")

In [ ]:
from pat2vec.util.config_pat2vec import config_class

config_obj = config_class(
    proj_name=PROJ_NAME,
    credentials_path=creds_filename,
    current_path_dir="",
    main_options={"epic_clinical_notes_appointments_annotations": True},
    batch_mode=True,
    verbosity=0,
    random_seed_val=22,
    testing=True,
    testing_elastic=True,
    dummy_medcat_model=True,
    use_controls=False,
    medcat=False,
    start_time=None,
    patient_id_column_name="client_idcode",
    annot_filter_options={},
    shuffle_pat_list=False,
    storage_backend="database",
    db_connection_string=db_connection_string,
    all_patient_list=patient_ids,
)

print(
    "pat2vec configuration created with epic_clinical_notes_appointments mode and database backend.",
)

In [ ]:
from pat2vec.main_pat2vec import main

try:
    pat2vec_obj = main(
        cogstack=True,
        use_filter=False,
        json_filter_path=None,
        random_seed_val=22,
        hostname=None,
        config_obj=config_obj,
    )
except FileNotFoundError as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: config path invalid. Error details: {e}.",
    ) from e
except ValueError as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: invalid configuration. Error details: {e}.",
    ) from e
except RuntimeError as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: initialization failure. Error details: {e}.",
    ) from e
except Exception as e:
    raise RuntimeError(
        f"Failed to initialize pipeline: unexpected error. Error details: {e}.",
    ) from e

print("pat2vec object initialized.")
print(f"Patient list: {pat2vec_obj.all_patient_list}")

In [ ]:
if not pat2vec_obj.all_patient_list:
    raise RuntimeError(
        "No patients in patient list after initialization. "
        "This indicates a critical failure in data loading or filtering.",
    )

print(f"Processing patient: {pat2vec_obj.all_patient_list[0]}")

try:
    pat2vec_obj.pat_maker(0)
except Exception as e:
    raise RuntimeError(
        f"Failed to process patient 0 with pat_maker: {e}. "
        "Critical error - pipeline failed to extract features.",
    ) from e

print("Patient feature extraction complete.")

In [ ]:
from pat2vec.util.helper_functions import get_all_features

all_features = get_all_features(config_obj)

if all_features.empty:
    raise RuntimeError(
        "FATAL ERROR: get_all_features returned an empty DataFrame. "
        "This indicates a critical failure in the pat2vec pipeline. "
        "No features were extracted or saved to the database.",
    )

print(f"Successfully retrieved {all_features.shape[0]} rows from database.")

In [ ]:
print("\n=== TABLE ERROR CHECK AFTER PAT_MAKER ===")

# Check pat_maker output above this cell for any of these strings.
# If any appear, the merge builders below will return empty data.
table_error_strings = [
    "no such table",
    "operationalerror",
    "table not found",
    "no table named",
]
print("Pat_maker output above this cell was scanned for database errors.")
print("If any of these strings appear in pat_maker output, the merge will fail:")
for s in table_error_strings:
    print(f"  - {s}")
print("")

In [ ]:
# === VECTOR VALIDATION ===
feature_cols = [c for c in all_features.columns if c.startswith("appointments_")]

assert len(feature_cols) > 0, "No feature columns found. Available columns: " + str(
    list(all_features.columns)
)

non_null_counts = all_features[feature_cols].notna().sum()
totally_empty = non_null_counts[non_null_counts == 0]

assert len(totally_empty) == 0, (
    f"The following feature columns are entirely null:\n"
    f"{list(totally_empty.index)}\n"
    "Vectorisation is silently failing — check the get method return value."
)

print("Feature columns (" + str(len(feature_cols)) + "): " + str(feature_cols))
print("Non-null counts per feature column:")
for col in sorted(feature_cols):
    val = all_features[col].notna().sum()
    print("  " + str(col) + ": " + str(val) + " non-null values")

In [ ]:
print("\n=== MERGE BUILDER FUNCTIONALITY TEST ===")

from pat2vec.util.post_processing_build_methods import (
    build_merged_epr_mct_annot_df,
)

merged_path = build_merged_epr_mct_annot_df(
    pat2vec_obj.all_patient_list,
    config_obj,
    overwrite=True,
)

assert (
    merged_path is not None
), "build_merged_epr_mct_annot_df returned None - expected a file path"

assert os.path.exists(merged_path), f"Merged file does not exist at {merged_path}"

merged_df = pd.read_csv(merged_path)
assert not merged_df.empty, (
    "Merged DataFrame is empty after pat_maker ran. "
    "pat_maker likely encountered a table error - check output above for "
    "no such table or OperationalError messages."
)

# Verify expected columns exist in the merged data
expected_columns = [
    "client_idcode",
    "document_guid",
    "annotation_description",
]
for col in expected_columns:
    assert col in merged_df.columns, f"Expected column {col} not found in merged data"

# Verify feature vector contains expected feature data
if len(merged_df) > 0 and "annotation_description" in merged_df.columns:
    non_null_annotations = merged_df["annotation_description"].dropna()
    assert len(non_null_annotations) > 0, (
        "Feature vector should contain annotation data but is empty. "
        "This indicates the merge builder may not be extracting features correctly."
    )

print(f"Merged data saved to: {merged_path}")
print(f"Shape: {merged_df.shape}")
print(f"Columns: {list(merged_df.columns)}")
if len(non_null_annotations) > 0:
    print("Sample of merged annotation_description values:")
    print(non_null_annotations.head())
else:
    print("Note: All annotations are empty/NAN - check pat_maker output above")

In [ ]:
print("\n=== DATABASE AND PROJECT CLEANUP ===")

try:
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print(f"Removed database: {DB_PATH}")
except Exception as e:
    raise RuntimeError(
        f"Failed to remove database file '{DB_PATH}': {e}. Critical error - cleanup incomplete.",
    ) from e

try:
    if os.path.exists(PROJ_NAME):
        shutil.rmtree(PROJ_NAME, ignore_errors=False)
        print(f"Removed project directory: {PROJ_NAME}")
except Exception as e:
    raise RuntimeError(
        f"Failed to remove '{PROJ_NAME}' directory: {e}. Critical error - cleanup incomplete.",
    ) from e

try:
    if os.path.exists(creds_filename):
        os.remove(creds_filename)
        print(f"Removed Elasticsearch credentials: {creds_filename}")
except Exception as e:
    raise RuntimeError(
        f"Failed to remove Elasticsearch credentials file '{creds_filename}': {e}. "
        "Critical error - cleanup incomplete.",
    ) from e

In [ ]:
print("\n=== FINAL VERIFICATION ===")

assert not os.path.exists(DB_PATH), "Database file still exists!"
assert not os.path.exists(PROJ_NAME), "Project directory still exists!"
assert not os.path.exists(
    creds_filename,
), "Elasticsearch credentials file still exists!"

print("All cleanup verified - no residual files remain.")
print("\n=== TEST SUCCESSFUL ===")